# Does Money Buy Wins? Predicting European Football Match Outcomes from Club Spending
### Traditional Machine Learning vs Deep Learning on Transfermarkt Data

**Author:** *[Your name]*, African Leadership University  
**Module:** Machine Learning, Summative Assignment (Model Training and Evaluation)

**Problem.** Clubs such as Manchester City have spent billions on transfers, and their rise to Premier League and UEFA Champions League (UCL) titles is often attributed to that spending. This project asks how far financial power (squad market value and net transfer spend) predicts match results, and how much it adds over sporting signals such as recent form and Elo strength ratings.

**Task.** Three-class classification of each match: `0` = away win, `1` = draw, `2` = home win.

**Why match-level?** There have only been about 70 UCL finals, far too few to model. Tens of thousands of league and European matches let us learn the money–results relationship reliably, and the UCL matches are then analysed as a subset and a case study.

**Notebook structure**
1. Setup and reproducibility
2. Data loading
3. Building leakage-free features (form, Elo, squad value, transfer spend)
4. Exploratory data analysis: does money buy points?
5. Temporal split and scaling
6. Evaluation and experiment-logging utilities
7. Traditional ML experiments (scikit-learn)
8. Deep learning experiments (TensorFlow/Keras with `tf.data`)
9. Experiment table
10. Final test-set evaluation
11. Error analysis, UCL subset and Manchester City case study
12. Saving models and artefacts
13. Conclusions

## 1. Setup and reproducibility
A single `SEED` drives Python, NumPy, TensorFlow and scikit-learn. `kagglehub` downloads the dataset; the other packages are preinstalled on Colab.

In [ ]:
!pip install -q kagglehub tabulate tensorboard

In [ ]:
import os, json, time, random, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sklearn
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

from sklearn.base import clone
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit, learning_curve
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             log_loss, roc_curve, confusion_matrix, ConfusionMatrixDisplay,
                             classification_report)

warnings.filterwarnings('ignore')
SEED = 42

def set_seeds(seed=SEED):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seeds()
for d in ['data', 'figures', 'models', 'logs', 'results']:
    os.makedirs(d, exist_ok=True)
sns.set_theme(style='whitegrid')
CLASS_NAMES = ['Away win', 'Draw', 'Home win']
print('TensorFlow', tf.__version__, '| scikit-learn', sklearn.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

## 2. Data loading
**Source:** *Football Data from Transfermarkt* (Kaggle, `davidcariboo/player-scores`), scraped from Transfermarkt.com and updated weekly. It is real, publicly available data, not synthetic. We use five linked tables:

| Table | Used for |
|---|---|
| `games` | match date, competition, teams and final score (the target) |
| `appearances` | which players each club used in each match |
| `player_valuations` | dated market values of every player |
| `transfers` | dated transfer fees between clubs |
| `clubs` | club names (for the Manchester City case study) |

The raw files are zipped into `data/` for the Google Drive folder and are **not** committed to GitHub. If the automatic download fails, download the dataset zip from Kaggle, unzip it into `data/player-scores/` and re-run.

In [ ]:
try:
    import kagglehub
    DATA_DIR = kagglehub.dataset_download('davidcariboo/player-scores')
except Exception as e:
    print('kagglehub download failed, using local folder:', e)
    DATA_DIR = 'data/player-scores'

def load(name):
    df = pd.read_csv(os.path.join(DATA_DIR, f'{name}.csv'), low_memory=False)
    print(f'{name:18s} {df.shape}')
    return df

games_raw = load('games')
appearances = load('appearances')
valuations = load('player_valuations')
transfers = load('transfers')
clubs = load('clubs')

shutil.make_archive('data/transfermarkt_raw', 'zip', DATA_DIR)
print('\nRaw matches:', f'{len(games_raw):,}')
games_raw.head()

## 3. Building leakage-free features
Every feature for a match uses **only information available before kick-off**. This is the most important design decision in the project: using, for example, the season's final league position or players signed later in the season would leak future information and inflate results.

**3.1 Clean matches and define the target.** Matches without a final score are dropped. The target is away win (0), draw (1) or home win (2).

In [ ]:
TARGET = 'result'
g = games_raw.dropna(subset=['home_club_id', 'away_club_id', 'home_club_goals', 'away_club_goals', 'date']).copy()
g['date'] = pd.to_datetime(g['date'])
g = g[g['home_club_id'] != g['away_club_id']]
for c in ['home_club_id', 'away_club_id', 'home_club_goals', 'away_club_goals']:
    g[c] = g[c].astype(int)
g = g.sort_values(['date', 'game_id']).reset_index(drop=True)
g[TARGET] = np.select([g.home_club_goals > g.away_club_goals, g.home_club_goals == g.away_club_goals], [2, 1], 0)
if 'competition_type' not in g.columns:
    comps = pd.read_csv(os.path.join(DATA_DIR, 'competitions.csv'))
    g = g.merge(comps[['competition_id', 'type']].rename(columns={'type': 'competition_type'}), on='competition_id', how='left')
print(f'{len(g):,} matches with scores, {g.date.min().date()} to {g.date.max().date()}')
print(g['competition_type'].value_counts())

**3.2 Club-level match history.** Each match becomes two rows, one per club, so rolling features can be computed per club across all competitions.
- `form_pts5`, `form_gd5`: mean points and goal difference over the club's previous 5 matches.
- `rest_days`: days since the club's previous match (fixture congestion), capped at 30.

`shift(1)` guarantees the current match is never included in its own features.

In [ ]:
base_cols = ['game_id', 'date', 'season', 'competition_id']
home = g[base_cols + ['home_club_id', 'away_club_id', 'home_club_goals', 'away_club_goals']].copy()
home.columns = base_cols + ['club_id', 'opp_id', 'gf', 'ga']; home['is_home'] = 1
away = g[base_cols + ['away_club_id', 'home_club_id', 'away_club_goals', 'home_club_goals']].copy()
away.columns = base_cols + ['club_id', 'opp_id', 'gf', 'ga']; away['is_home'] = 0
long = pd.concat([home, away], ignore_index=True)
long['points'] = np.select([long.gf > long.ga, long.gf == long.ga], [3, 1], 0)
long['gd'] = long.gf - long.ga
long = long.sort_values(['club_id', 'date', 'game_id']).reset_index(drop=True)

grp = long.groupby('club_id')
prev5 = lambda s: s.shift(1).rolling(5, min_periods=1).mean()
long['form_pts5'] = grp['points'].transform(prev5)
long['form_gd5'] = grp['gd'].transform(prev5)
long['n_prev_games'] = grp.cumcount()
long['rest_days'] = grp['date'].diff().dt.days.clip(upper=30).fillna(30)
long[['club_id', 'date', 'points', 'form_pts5', 'form_gd5', 'rest_days']].head(8)

**3.3 Squad market value (money, part 1).** For every player appearance, we look up that player's most recent market valuation *strictly before* the match date (`merge_asof`, valuations older than two years ignored). Summing over players used gives a club's *matchday squad value*. The feature `sq_value5` is the mean matchday value over the club's **previous** five matches, so it reflects the squad the club has been fielding, not the line-up of the match being predicted.

In [ ]:
vdate = 'date' if 'date' in valuations.columns else 'datetime'
val = valuations[['player_id', vdate, 'market_value_in_eur']].rename(columns={vdate: 'val_date'}).dropna()
val['val_date'] = pd.to_datetime(val['val_date'])
val = val.sort_values('val_date')

apps = appearances[['game_id', 'player_id', 'player_club_id', 'date']].dropna().copy()
apps['date'] = pd.to_datetime(apps['date'])
apps = pd.merge_asof(apps.sort_values('date'), val, left_on='date', right_on='val_date', by='player_id',
                     direction='backward', allow_exact_matches=False, tolerance=pd.Timedelta(days=730))
mday = (apps.groupby(['game_id', 'player_club_id'])['market_value_in_eur'].sum(min_count=1)
            .rename('matchday_value').reset_index().rename(columns={'player_club_id': 'club_id'}))
mday['club_id'] = mday['club_id'].astype(int)

long = long.merge(mday, on=['game_id', 'club_id'], how='left').sort_values(['club_id', 'date', 'game_id']).reset_index(drop=True)
long['sq_value5'] = long.groupby('club_id')['matchday_value'].transform(prev5)
print('Appearances with a valuation:', f"{apps['market_value_in_eur'].notna().mean():.1%}")
print('Club-matches with a squad value:', f"{long['sq_value5'].notna().mean():.1%}")

**3.4 Transfer spending (money, part 2).** From the transfers table we compute, for each club at each match date, spending over the **previous** 365 days and 3 years:
- `net_spend_1y`, `net_spend_3y`: fees paid minus fees received;
- `gross_spend_1y`: fees paid only.

Implemented with cumulative sums per club and `merge_asof` lookups, again strictly before the match date. Unknown fees (loans, free transfers) count as zero.

In [ ]:
tr = transfers[['transfer_date', 'from_club_id', 'to_club_id', 'transfer_fee']].copy()
tr['transfer_date'] = pd.to_datetime(tr['transfer_date'], errors='coerce')
tr['transfer_fee'] = pd.to_numeric(tr['transfer_fee'], errors='coerce').fillna(0)
tr = tr.dropna(subset=['transfer_date'])

buys = tr.dropna(subset=['to_club_id']).rename(columns={'to_club_id': 'club_id', 'transfer_date': 'date'})
sells = tr.dropna(subset=['from_club_id']).rename(columns={'from_club_id': 'club_id', 'transfer_date': 'date'})
buys = buys[['club_id', 'date', 'transfer_fee']].assign(net=lambda d: d.transfer_fee, gross=lambda d: d.transfer_fee)
sells = sells[['club_id', 'date', 'transfer_fee']].assign(net=lambda d: -d.transfer_fee, gross=0.0)
ev = pd.concat([buys, sells]).astype({'club_id': int}).groupby(['club_id', 'date'])[['net', 'gross']].sum().reset_index()
ev = ev.sort_values(['club_id', 'date'])
ev[['cum_net', 'cum_gross']] = ev.groupby('club_id')[['net', 'gross']].cumsum()
ev = ev.sort_values('date')

def cum_before(dates):
    q = pd.DataFrame({'club_id': long['club_id'].values, 'date': dates.values, 'row': np.arange(len(long))}).sort_values('date')
    out = pd.merge_asof(q, ev[['club_id', 'date', 'cum_net', 'cum_gross']], on='date', by='club_id',
                        direction='backward', allow_exact_matches=False)
    return out.sort_values('row')[['cum_net', 'cum_gross']].fillna(0).values

now = cum_before(long['date'])
y1 = cum_before(long['date'] - pd.Timedelta(days=365))
y3 = cum_before(long['date'] - pd.Timedelta(days=3 * 365))
long['net_spend_1y'] = now[:, 0] - y1[:, 0]
long['gross_spend_1y'] = now[:, 1] - y1[:, 1]
long['net_spend_3y'] = now[:, 0] - y3[:, 0]
long[['net_spend_1y', 'gross_spend_1y', 'net_spend_3y']].describe().T

**3.5 Elo rating (sporting strength).** A classic rating system updated after every match, including a home advantage and a margin-of-victory multiplier. Each match receives the ratings *before* it is played. Elo summarises long-run sporting strength, so comparing models with and without it shows whether money adds information beyond past results.

*Limitation:* clubs in different leagues only meet in European competition, so Elo is not perfectly comparable across leagues. Market value partly corrects for this.

In [ ]:
def compute_elo(df, k=20, home_adv=60, init=1500):
    elo, pre_h, pre_a = {}, np.empty(len(df)), np.empty(len(df))
    for i, (h, a, hg, ag) in enumerate(zip(df.home_club_id, df.away_club_id, df.home_club_goals, df.away_club_goals)):
        rh, ra = elo.get(h, init), elo.get(a, init)
        pre_h[i], pre_a[i] = rh, ra
        exp_h = 1 / (1 + 10 ** ((ra - rh - home_adv) / 400))
        score = 1.0 if hg > ag else 0.5 if hg == ag else 0.0
        delta = k * (np.log(abs(hg - ag) + 1) + 1) * (score - exp_h)
        elo[h], elo[a] = rh + delta, ra - delta
    return pre_h, pre_a, elo

g['home_elo'], g['away_elo'], FINAL_ELO = compute_elo(g)
pd.Series(FINAL_ELO).sort_values(ascending=False).head(10).rename('final Elo').to_frame().join(
    clubs.set_index('club_id')['name'])

**3.6 Assemble the match-level table.** Home and away club features are joined back to each match, and difference features are created (home minus away). Money features are log-transformed because squad values and fees are extremely right-skewed (a handful of clubs spend orders of magnitude more than the rest); a signed log handles negative net spend. We keep domestic-league and European-cup matches, and require both clubs to have at least five previous matches and a known squad value.

In [ ]:
CLUB_FEATS = ['form_pts5', 'form_gd5', 'rest_days', 'sq_value5', 'net_spend_1y', 'gross_spend_1y', 'net_spend_3y', 'n_prev_games']
H = long[long.is_home == 1].set_index('game_id')[CLUB_FEATS].add_prefix('home_')
A = long[long.is_home == 0].set_index('game_id')[CLUB_FEATS].add_prefix('away_')
m = g.set_index('game_id').join(H).join(A).reset_index()

slog = lambda x: np.sign(x) * np.log1p(np.abs(x) / 1e6)       # signed log of euros, in millions
for s in ['home', 'away']:
    m[f'{s}_log_value'] = np.log1p(m[f'{s}_sq_value5'] / 1e6)
    for c in ['net_spend_1y', 'gross_spend_1y', 'net_spend_3y']:
        m[f'{s}_{c}_slog'] = slog(m[f'{s}_{c}'])
m['value_diff'] = m.home_log_value - m.away_log_value
m['net_spend_1y_diff'] = m.home_net_spend_1y_slog - m.away_net_spend_1y_slog
m['net_spend_3y_diff'] = m.home_net_spend_3y_slog - m.away_net_spend_3y_slog
m['elo_diff'] = m.home_elo - m.away_elo
m['form_pts_diff'] = m.home_form_pts5 - m.away_form_pts5
m['form_gd_diff'] = m.home_form_gd5 - m.away_form_gd5
m['is_european'] = (m['competition_type'] == 'international_cup').astype(int)
m['is_ucl'] = (m['competition_id'] == 'CL').astype(int)

n0 = len(m)
m = m[m.competition_type.isin(['domestic_league', 'international_cup'])]
m = m[(m.home_n_prev_games >= 5) & (m.away_n_prev_games >= 5)]
m = m.dropna(subset=['home_log_value', 'away_log_value']).sort_values(['date', 'game_id']).reset_index(drop=True)
print(f'{n0:,} matches -> {len(m):,} modelling rows after filtering')
print('Class balance:', m[TARGET].value_counts(normalize=True).sort_index().round(3).rename(dict(enumerate(CLASS_NAMES))).to_dict())
print('UCL matches:', int(m.is_ucl.sum()))

In [ ]:
MONEY = ['home_log_value', 'away_log_value', 'value_diff',
         'home_net_spend_1y_slog', 'away_net_spend_1y_slog', 'net_spend_1y_diff',
         'home_net_spend_3y_slog', 'away_net_spend_3y_slog', 'net_spend_3y_diff',
         'home_gross_spend_1y_slog', 'away_gross_spend_1y_slog']
SPORT = ['home_elo', 'away_elo', 'elo_diff', 'home_form_pts5', 'away_form_pts5', 'form_pts_diff',
         'home_form_gd5', 'away_form_gd5', 'form_gd_diff', 'home_rest_days', 'away_rest_days']
CONTEXT = ['is_european', 'is_ucl']
FEATURE_SETS = {'money': MONEY + CONTEXT, 'sport': SPORT + CONTEXT, 'full': MONEY + SPORT + CONTEXT}
{k: len(v) for k, v in FEATURE_SETS.items()}

## 4. Exploratory data analysis: does money buy points?

In [ ]:
counts = m[TARGET].value_counts().sort_index()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(CLASS_NAMES, counts.values, color=['#C44E52', '#8C8C8C', '#4C72B0'])
for i, v in enumerate(counts.values):
    axes[0].text(i, v, f'{v:,}\n({v / counts.sum():.1%})', ha='center', va='bottom')
axes[0].set_ylim(0, counts.max() * 1.2); axes[0].set_title('Match outcomes (all modelling rows)')
by_comp = m.groupby('is_european')[TARGET].value_counts(normalize=True).unstack().rename(columns=dict(enumerate(CLASS_NAMES)))
by_comp.index = ['Domestic league', 'European cup']
by_comp.plot(kind='bar', stacked=True, ax=axes[1], color=['#C44E52', '#8C8C8C', '#4C72B0'], rot=0)
axes[1].set_title('Outcome mix by competition type'); axes[1].legend(loc='lower right')
plt.tight_layout(); plt.savefig('figures/class_balance.png', dpi=150); plt.show()

**Interpretation.** Home wins are the most common outcome and draws the least, but the classes are only moderately imbalanced. Draws are expected to be the hardest class: they are the "in-between" outcome with no strong feature signature.

In [ ]:
league = long.merge(g[['game_id', 'competition_type']], on='game_id')
league = league[league.competition_type == 'domestic_league']
cs = league.groupby(['club_id', 'season']).agg(ppg=('points', 'mean'), value=('sq_value5', 'mean'),
                                               spend=('net_spend_1y', 'mean'), n=('points', 'size')).dropna()
cs = cs[cs.n >= 15]
cs['log_value'] = np.log1p(cs.value / 1e6)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.scatterplot(data=cs, x='log_value', y='ppg', s=10, alpha=0.4, ax=axes[0])
axes[0].set(xlabel='log(1 + mean squad value, €m)', ylabel='Points per game',
            title=f"Squad value vs points (Spearman ρ = {cs[['log_value', 'ppg']].corr('spearman').iloc[0, 1]:.2f})")
sns.scatterplot(x=slog(cs.spend), y=cs.ppg, s=10, alpha=0.4, ax=axes[1])
axes[1].set(xlabel='signed log net spend, previous 12 months (€m)', ylabel='Points per game',
            title=f"Net spend vs points (Spearman ρ = {pd.Series(slog(cs.spend)).corr(cs.ppg, method='spearman'):.2f})")
plt.tight_layout(); plt.savefig('figures/money_vs_points.png', dpi=150); plt.show()

**Interpretation (check against your output).** Each dot is one club-season in a domestic league. If squad value correlates much more strongly with points than net spend does, that supports a key argument for the report: *accumulated* talent (market value) matters more than a single window's spending, which is noisy (a club may spend heavily to replace stars it sold).

In [ ]:
corr = m[FEATURE_SETS['full'] + [TARGET]].corr(method='spearman')[TARGET].drop(TARGET).sort_values()
corr.plot(kind='barh', figsize=(7, 8), color=['#C44E52' if v < 0 else '#4C72B0' for v in corr])
plt.title('Spearman correlation with outcome (0=away, 1=draw, 2=home)')
plt.tight_layout(); plt.savefig('figures/feature_correlation.png', dpi=150); plt.show()

## 5. Temporal split and scaling
Random splitting would let the model train on matches played *after* the ones it is tested on. Instead, matches are ordered by date: the earliest **70%** are training, the next **15%** validation (model selection and tuning) and the most recent **15%** test (used once, in Section 10). This mimics real use: predicting future matches from the past. Hyperparameter searches also use `TimeSeriesSplit` inside the training period.

Continuous features are standardised with statistics from the training period only; binary flags are left as 0/1.

In [ ]:
i_tr, i_va = int(len(m) * 0.70), int(len(m) * 0.85)
train_df, val_df, test_df = m.iloc[:i_tr], m.iloc[i_tr:i_va], m.iloc[i_va:]
for name, d in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(f"{name:5s}: {len(d):7,} rows | {d.date.min().date()} to {d.date.max().date()} | "
          f"{d[TARGET].value_counts(normalize=True).sort_index().round(3).tolist()}")
y_train, y_val, y_test = (d[TARGET].values for d in (train_df, val_df, test_df))

def scale_sets(cols):
    to_scale = [c for c in cols if train_df[c].nunique() > 2]
    scaler = StandardScaler().fit(train_df[to_scale])
    def transform(d):
        out = d[cols].astype('float32').copy()
        out[to_scale] = scaler.transform(d[to_scale])
        return out.values.astype('float32')
    return {'train': transform(train_df), 'val': transform(val_df), 'test': transform(test_df),
            'scaler': scaler, 'scaled_cols': to_scale, 'columns': cols}

DATA = {k: scale_sets(v) for k, v in FEATURE_SETS.items()}
{k: v['train'].shape for k, v in DATA.items()}

## 6. Evaluation and experiment-logging utilities
All experiments are scored identically on the validation set:
- **macro ROC-AUC** (one-vs-rest), the primary metric used to rank models, since it measures how well each outcome is ranked regardless of class weighting;
- **log loss**, which rewards well-calibrated probabilities (what a bookmaker or analyst actually needs);
- accuracy, macro precision, macro recall, macro F1 and **draw recall** (the hardest class).

`record()` appends each run to `results/experiment_log.csv` and writes the same numbers to TensorBoard (`logs/<exp_id>/eval`), so every row of the experiment table is traceable to a logged run. `auc_gap` (train minus validation ROC-AUC) is the overfitting indicator.

In [ ]:
RESULTS, MODELS, VAL_PROBS, HISTORIES, META = [], {}, {}, {}, {}
TR_IDX = np.random.RandomState(SEED).choice(len(y_train), size=min(40_000, len(y_train)), replace=False)

def metrics_mc(y, P):
    yhat = P.argmax(1)
    return {'accuracy': accuracy_score(y, yhat),
            'precision_macro': precision_score(y, yhat, average='macro', zero_division=0),
            'recall_macro': recall_score(y, yhat, average='macro'),
            'f1_macro': f1_score(y, yhat, average='macro'),
            'draw_recall': recall_score(y, yhat, labels=[1], average='macro'),
            'roc_auc_macro': roc_auc_score(y, P, multi_class='ovr', average='macro'),
            'log_loss': log_loss(y, np.clip(P, 1e-7, 1), labels=[0, 1, 2])}

def log_tb_scalars(exp_id, metrics):
    writer = tf.summary.create_file_writer(f'logs/{exp_id}/eval')
    with writer.as_default():
        for k, v in metrics.items():
            tf.summary.scalar(k, float(v), step=0)
    writer.flush()

def record(exp_id, paradigm, model_name, features, settings, P_val, train_auc, train_time, notes=''):
    global RESULTS
    mt = metrics_mc(y_val, P_val)
    mt.update({'train_roc_auc': train_auc, 'auc_gap': train_auc - mt['roc_auc_macro']})
    row = {'exp_id': exp_id, 'paradigm': paradigm, 'model': model_name, 'features': features,
           'split': 'temporal 70/15/15; metrics on validation', 'settings': json.dumps(settings, default=str),
           **{k: round(float(v), 4) for k, v in mt.items()}, 'train_time_s': round(train_time, 1), 'notes': notes}
    RESULTS = [r for r in RESULTS if r['exp_id'] != exp_id] + [row]
    log_tb_scalars(exp_id, mt)
    pd.DataFrame(RESULTS).to_csv('results/experiment_log.csv', index=False)
    print(f"{exp_id}: ROC-AUC={mt['roc_auc_macro']:.4f} logloss={mt['log_loss']:.4f} acc={mt['accuracy']:.3f} "
          f"F1m={mt['f1_macro']:.3f} draw_rec={mt['draw_recall']:.3f} gap={mt['auc_gap']:.3f} [{train_time:.0f}s]")

## 7. Traditional machine learning experiments (scikit-learn)
Each run changes one thing relative to an earlier one:

| ID | Change being tested |
|---|---|
| ML-1 | Logistic Regression on **money features only**: how much do finances alone explain? |
| ML-2 | Logistic Regression on **sporting features only** (Elo, form, rest) |
| ML-3 | Logistic Regression on **all features**: does money add information beyond sport? |
| ML-4 | ML-3 + balanced class weights (effect on draws) |
| ML-5 | Random Forest, default (non-linear, high variance) |
| ML-6 | Random Forest, tuned (`RandomizedSearchCV`, `TimeSeriesSplit`) |
| ML-7 | Histogram Gradient Boosting, default |
| ML-8 | Histogram Gradient Boosting, tuned |

In [ ]:
def run_sklearn(exp_id, model, features, settings, notes=''):
    Xtr, Xva = DATA[features]['train'], DATA[features]['val']
    t0 = time.time(); model.fit(Xtr, y_train); tt = time.time() - t0
    P_val = model.predict_proba(Xva)
    train_auc = roc_auc_score(y_train[TR_IDX], model.predict_proba(Xtr[TR_IDX]), multi_class='ovr')
    MODELS[exp_id], VAL_PROBS[exp_id] = model, P_val
    META[exp_id] = {'paradigm': 'Traditional ML', 'features': features}
    record(exp_id, 'Traditional ML', type(model).__name__, features, settings, P_val, train_auc, tt, notes)
    return model

### ML-1 to ML-3: the feature-set ablation
This is the core of the research question. Comparing money-only, sport-only and combined models shows how much predictive information finances carry, and whether they are redundant with past performance (rich clubs are also usually high-Elo clubs).

In [ ]:
for eid, fs, note in [('ML-1', 'money', 'Money features only'), ('ML-2', 'sport', 'Sporting features only'),
                      ('ML-3', 'full', 'Money + sport')]:
    run_sklearn(eid, LogisticRegression(max_iter=3000, random_state=SEED), fs, {'C': 1.0, 'class_weight': None}, note)

### ML-4: class weighting
Unweighted models almost never predict draws because a draw is rarely the single most likely outcome. Balanced weights should raise draw recall at the cost of accuracy and log loss, a trade-off worth quantifying.

In [ ]:
run_sklearn('ML-4', LogisticRegression(max_iter=3000, class_weight='balanced', random_state=SEED), 'full',
            {'C': 1.0, 'class_weight': 'balanced'}, 'Class-weighted');

### ML-5 and ML-6: Random Forest, default then tuned
Default forests grow full-depth trees and memorise noisy football results (expect a large `auc_gap`). The search constrains depth and leaf size; scoring is ROC-AUC with time-ordered folds.

In [ ]:
TSCV = TimeSeriesSplit(n_splits=3)
run_sklearn('ML-5', RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=SEED), 'full',
            {'n_estimators': 200, 'max_depth': None, 'min_samples_leaf': 1}, 'Default RF (expect overfitting)');

rf_grid = {'n_estimators': [200, 400], 'max_depth': [6, 10, 14, None], 'min_samples_leaf': [5, 20, 50, 100],
           'max_features': ['sqrt', 0.5], 'class_weight': [None, 'balanced_subsample']}
rf_search = RandomizedSearchCV(RandomForestClassifier(n_jobs=-1, random_state=SEED), rf_grid, n_iter=10,
                               scoring='roc_auc_ovr', cv=TSCV, random_state=SEED, verbose=1)
rf_search.fit(DATA['full']['train'], y_train)
pd.DataFrame(rf_search.cv_results_).sort_values('rank_test_score').to_csv('results/rf_search.csv', index=False)
print('Best RF:', rf_search.best_params_, '| CV ROC-AUC:', round(rf_search.best_score_, 4))
run_sklearn('ML-6', RandomForestClassifier(**rf_search.best_params_, n_jobs=-1, random_state=SEED), 'full',
            rf_search.best_params_, 'Tuned RF');

### ML-7 and ML-8: Histogram Gradient Boosting, default then tuned

In [ ]:
run_sklearn('ML-7', HistGradientBoostingClassifier(random_state=SEED), 'full',
            {'learning_rate': 0.1, 'max_iter': 100, 'max_leaf_nodes': 31}, 'Default HGB');

hgb_grid = {'learning_rate': [0.02, 0.05, 0.1], 'max_iter': [200, 400], 'max_leaf_nodes': [7, 15, 31],
            'l2_regularization': [0.0, 1.0, 10.0], 'min_samples_leaf': [50, 100, 200]}
hgb_search = RandomizedSearchCV(HistGradientBoostingClassifier(early_stopping=True, random_state=SEED), hgb_grid,
                                n_iter=15, scoring='roc_auc_ovr', cv=TSCV, random_state=SEED, verbose=1)
hgb_search.fit(DATA['full']['train'], y_train)
pd.DataFrame(hgb_search.cv_results_).sort_values('rank_test_score').to_csv('results/hgb_search.csv', index=False)
print('Best HGB:', hgb_search.best_params_, '| CV ROC-AUC:', round(hgb_search.best_score_, 4))
run_sklearn('ML-8', HistGradientBoostingClassifier(**hgb_search.best_params_, early_stopping=True, random_state=SEED),
            'full', hgb_search.best_params_, 'Tuned HGB');

In [ ]:
ml_res = pd.DataFrame(RESULTS).query("paradigm == 'Traditional ML'")
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ml_res.plot(x='exp_id', y='roc_auc_macro', kind='bar', ax=axes[0], rot=0, legend=False, color='#4C72B0')
axes[0].set_ylim(0.5, ml_res.roc_auc_macro.max() + 0.03); axes[0].set_title('Validation macro ROC-AUC')
ml_res.plot(x='exp_id', y='log_loss', kind='bar', ax=axes[1], rot=0, legend=False, color='#DD8452')
axes[1].set_ylim(ml_res.log_loss.min() - 0.03, ml_res.log_loss.max() + 0.01); axes[1].set_title('Validation log loss (lower is better)')
plt.tight_layout(); plt.savefig('figures/ml_comparison.png', dpi=150); plt.show()
ml_res[['exp_id', 'model', 'features', 'roc_auc_macro', 'log_loss', 'accuracy', 'f1_macro', 'draw_recall', 'auc_gap']]

**Interpretation (write after running).** Compare ML-1, ML-2 and ML-3: if money-only is close to sport-only, finances alone carry most of the predictive signal; if ML-3 is barely better than either, the two are largely redundant (money buys the players who produce the Elo). ML-4 should show higher draw recall but worse log loss. ML-5 vs ML-6 shows variance reduction through tuning. If the tuned boosting model is only slightly better than logistic regression, the relationship is close to additive in log-money and Elo difference.

## 8. Deep learning experiments (TensorFlow/Keras with `tf.data`)
All networks read data through `tf.data` pipelines (shuffle, batch, prefetch), output a 3-way softmax, and use EarlyStopping on validation loss (best weights restored), ReduceLROnPlateau and a TensorBoard callback.

| ID | Change being tested |
|---|---|
| DL-1 | Sequential MLP 64-32, Adam 1e-3 (baseline) |
| DL-2 | + class weights |
| DL-3 | Deeper 256-128-64 + BatchNorm + Dropout 0.3 |
| DL-4 | DL-3 + L2 weight decay 1e-4 |
| DL-5 | DL-4 with learning rate 1e-2 |
| DL-6 | DL-4 with learning rate 1e-4 |
| DL-7 | DL-4 with batch size 2048 |
| DL-8 | Functional API: numeric inputs + learned **competition embedding** |

In [ ]:
CLASS_WEIGHTS = dict(enumerate(compute_class_weight('balanced', classes=np.array([0, 1, 2]), y=y_train)))
print('Class weights:', {CLASS_NAMES[k]: round(v, 3) for k, v in CLASS_WEIGHTS.items()})

def take(X, idx):
    return {k: v[idx] for k, v in X.items()} if isinstance(X, dict) else X[idx]

def make_ds(X, y=None, batch_size=512, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices(X if y is None else (X, y.astype('int32')))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(y), seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

def compile_model(model, lr):
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr), loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

def build_mlp(input_dim, units=(64, 32), dropout=0.0, batchnorm=False, l2=0.0, lr=1e-3):
    reg = regularizers.l2(l2) if l2 > 0 else None
    model = keras.Sequential(name='mlp')
    model.add(keras.Input(shape=(input_dim,)))
    for u in units:
        model.add(layers.Dense(u, kernel_regularizer=reg))
        if batchnorm:
            model.add(layers.BatchNormalization())
        model.add(layers.Activation('relu'))
        if dropout > 0:
            model.add(layers.Dropout(dropout))
    model.add(layers.Dense(3, activation='softmax'))
    return compile_model(model, lr)

def plot_history(exp_id, h):
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
    for ax, k in zip(axes, ['loss', 'accuracy']):
        ax.plot(h[k], label='train'); ax.plot(h['val_' + k], label='validation')
        ax.set_title(f'{exp_id}: {k}'); ax.set_xlabel('epoch'); ax.legend()
    plt.tight_layout(); plt.savefig(f'figures/learning_curve_{exp_id}.png', dpi=150); plt.show()

def run_keras(exp_id, build_fn, features, settings, batch_size=512, epochs=50, class_weight=None, notes=''):
    set_seeds()
    model = build_fn()
    Xtr, Xva = DATA[features]['train'], DATA[features]['val']
    callbacks = [keras.callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True),
                 keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6),
                 keras.callbacks.TensorBoard(log_dir=f'logs/{exp_id}')]
    t0 = time.time()
    hist = model.fit(make_ds(Xtr, y_train, batch_size, shuffle=True), validation_data=make_ds(Xva, y_val, batch_size),
                     epochs=epochs, class_weight=class_weight, callbacks=callbacks, verbose=2)
    tt = time.time() - t0
    P_val = model.predict(make_ds(Xva, batch_size=4096), verbose=0)
    P_tr = model.predict(make_ds(take(Xtr, TR_IDX), batch_size=4096), verbose=0)
    s = dict(settings, batch_size=batch_size, class_weight=class_weight is not None,
             epochs_run=len(hist.history['loss']), params=model.count_params())
    MODELS[exp_id], VAL_PROBS[exp_id], HISTORIES[exp_id] = model, P_val, hist.history
    META[exp_id] = {'paradigm': 'Deep Learning', 'features': features}
    record(exp_id, 'Deep Learning', model.name, features, s, P_val,
           roc_auc_score(y_train[TR_IDX], P_tr, multi_class='ovr'), tt, notes)
    plot_history(exp_id, hist.history)
    return model

N_IN = DATA['full']['train'].shape[1]
DEEP = dict(units=(256, 128, 64), dropout=0.3, batchnorm=True)
DEEP_S = {'units': [256, 128, 64], 'dropout': 0.3, 'batchnorm': True}

### DL-1 and DL-2: baseline MLP, without and with class weights

In [ ]:
run_keras('DL-1', lambda: build_mlp(N_IN), 'full', {'units': [64, 32], 'lr': 1e-3}, notes='Baseline MLP');

In [ ]:
run_keras('DL-2', lambda: build_mlp(N_IN), 'full', {'units': [64, 32], 'lr': 1e-3},
          class_weight=CLASS_WEIGHTS, notes='+ class weights');

### DL-3 and DL-4: capacity, then regularisation
Football results are noisy, so extra capacity mostly fits noise. Watch whether validation loss starts rising while training loss keeps falling, and whether L2 narrows that gap.

In [ ]:
run_keras('DL-3', lambda: build_mlp(N_IN, lr=1e-3, **DEEP), 'full', dict(DEEP_S, lr=1e-3), notes='Deeper + BN + dropout');

In [ ]:
run_keras('DL-4', lambda: build_mlp(N_IN, l2=1e-4, lr=1e-3, **DEEP), 'full', dict(DEEP_S, l2=1e-4, lr=1e-3), notes='+ L2');

### DL-5 and DL-6: learning-rate sensitivity

In [ ]:
run_keras('DL-5', lambda: build_mlp(N_IN, l2=1e-4, lr=1e-2, **DEEP), 'full', dict(DEEP_S, l2=1e-4, lr=1e-2), notes='High LR');

In [ ]:
run_keras('DL-6', lambda: build_mlp(N_IN, l2=1e-4, lr=1e-4, **DEEP), 'full', dict(DEEP_S, l2=1e-4, lr=1e-4), notes='Low LR');

### DL-7: large batch size
Larger batches give smoother gradients and faster epochs but fewer updates per epoch; without raising the learning rate this usually slows convergence.

In [ ]:
run_keras('DL-7', lambda: build_mlp(N_IN, l2=1e-4, lr=1e-3, **DEEP), 'full', dict(DEEP_S, l2=1e-4, lr=1e-3),
          batch_size=2048, notes='Batch size 2048');

### DL-8: Functional API with a competition embedding
Home advantage, draw rates and the value of money differ between competitions (for example, European knockout ties versus a domestic league). A learned embedding for `competition_id` lets the network adjust for this. Competitions not seen in training map to a shared "unknown" index. Multiple named inputs require the Functional API.

In [ ]:
COMP_INDEX = {c: i + 1 for i, c in enumerate(sorted(train_df['competition_id'].unique()))}
def fn_inputs(d, split):
    return {'num': DATA['full'][split], 'competition': d['competition_id'].map(COMP_INDEX).fillna(0).astype('int32').values.reshape(-1, 1)}
DATA['fn'] = {'train': fn_inputs(train_df, 'train'), 'val': fn_inputs(val_df, 'val'), 'test': fn_inputs(test_df, 'test'),
              'scaler': DATA['full']['scaler'], 'scaled_cols': DATA['full']['scaled_cols'], 'columns': FEATURE_SETS['full']}

def build_functional(num_dim, n_comp, emb_dim=4, units=(128, 64), dropout=0.3, l2=1e-4, lr=1e-3):
    reg = regularizers.l2(l2)
    inputs = {'num': keras.Input(shape=(num_dim,), name='num'),
              'competition': keras.Input(shape=(1,), name='competition', dtype='int32')}
    emb = layers.Flatten()(layers.Embedding(n_comp + 1, emb_dim, name='competition_embedding')(inputs['competition']))
    x = layers.Concatenate()([inputs['num'], emb])
    for u in units:
        x = layers.Dense(u, kernel_regularizer=reg)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.Dropout(dropout)(x)
    out = layers.Dense(3, activation='softmax', name='outcome')(x)
    return compile_model(keras.Model(inputs=inputs, outputs=out, name='functional_comp_embedding'), lr)

build_functional(N_IN, len(COMP_INDEX)).summary()
run_keras('DL-8', lambda: build_functional(N_IN, len(COMP_INDEX)), 'fn',
          {'api': 'functional', 'emb_dim': 4, 'units': [128, 64], 'dropout': 0.3, 'l2': 1e-4, 'lr': 1e-3},
          notes='Competition embedding');

**Interpretation (write after running).** Use the learning curves: diverging training and validation loss indicates variance (overfitting); both curves flattening at a similar level indicates a bias or information ceiling. In football that ceiling is real, because match outcomes contain a large irreducible luck component. Compare DL-5 (jagged, unstable curves) with DL-6 (slow descent), and DL-8 with DL-4 (does knowing the competition help?).

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs

## 9. Experiment table
The full log of all runs, generated by `record()` and mirrored in TensorBoard (`logs/<exp_id>/eval`). Metrics are on the **validation** period.

In [ ]:
COLS = ['exp_id', 'paradigm', 'model', 'features', 'settings', 'split', 'roc_auc_macro', 'log_loss', 'accuracy',
        'precision_macro', 'recall_macro', 'f1_macro', 'draw_recall', 'train_roc_auc', 'auc_gap', 'train_time_s', 'notes']
exp_table = pd.DataFrame(RESULTS)[COLS]
exp_table.to_csv('results/experiment_table.csv', index=False)
with open('results/experiment_table.md', 'w') as f:
    f.write(exp_table.to_markdown(index=False))
pd.set_option('display.max_colwidth', 80)
exp_table

## 10. Final evaluation on the held-out test period
The best traditional and best deep-learning models (by validation macro ROC-AUC) are evaluated once on the most recent 15% of matches. Two simple reference points are included: **always predict a home win**, and **the more valuable squad wins** (a pure "money decides" rule).

In [ ]:
res = pd.DataFrame(RESULTS)
best_ml_id = res.query("paradigm == 'Traditional ML'").sort_values('roc_auc_macro', ascending=False).iloc[0]['exp_id']
best_dl_id = res.query("paradigm == 'Deep Learning'").sort_values('roc_auc_macro', ascending=False).iloc[0]['exp_id']
print('Best traditional:', best_ml_id, '| Best deep learning:', best_dl_id)

def predict(exp_id, split):
    X = DATA[META[exp_id]['features']][split]
    if META[exp_id]['paradigm'] == 'Deep Learning':
        return MODELS[exp_id].predict(make_ds(X, batch_size=4096), verbose=0)
    return MODELS[exp_id].predict_proba(X)

P_TEST = {eid: predict(eid, 'test') for eid in ['ML-1', best_ml_id, best_dl_id]}
rows = [{'model': f'{eid} ({res.set_index("exp_id").loc[eid, "notes"]})', **metrics_mc(y_test, P)} for eid, P in P_TEST.items()]
hard = lambda pred: {'accuracy': accuracy_score(y_test, pred), 'f1_macro': f1_score(y_test, pred, average='macro')}
rows.append({'model': 'Rule: always home win', **hard(np.full(len(y_test), 2))})
rows.append({'model': 'Rule: richer squad wins', **hard(np.where(test_df.value_diff.values >= 0, 2, 0))})
test_table = pd.DataFrame(rows).round(4)
test_table.to_csv('results/test_results.csv', index=False)
test_table

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, eid in zip(axes, [best_ml_id, best_dl_id]):
    ConfusionMatrixDisplay(confusion_matrix(y_test, P_TEST[eid].argmax(1)), display_labels=CLASS_NAMES).plot(
        ax=ax, cmap='Blues', values_format=',')
    ax.set_title(f'{eid} (test)')
plt.tight_layout(); plt.savefig('figures/confusion_matrices_test.png', dpi=150); plt.show()
for eid in [best_ml_id, best_dl_id]:
    print(f'--- {eid} ---')
    print(classification_report(y_test, P_TEST[eid].argmax(1), target_names=CLASS_NAMES, digits=3))

In [ ]:
Y_bin = label_binarize(y_test, classes=[0, 1, 2])
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, eid in zip(axes, [best_ml_id, best_dl_id]):
    for k, name in enumerate(CLASS_NAMES):
        fpr, tpr, _ = roc_curve(Y_bin[:, k], P_TEST[eid][:, k])
        ax.plot(fpr, tpr, label=f'{name} (AUC={roc_auc_score(Y_bin[:, k], P_TEST[eid][:, k]):.3f})')
    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.set(xlabel='False positive rate', ylabel='True positive rate', title=f'One-vs-rest ROC: {eid} (test)'); ax.legend()
plt.tight_layout(); plt.savefig('figures/roc_test.png', dpi=150); plt.show()

### Learning curve for the best traditional model
Training size is varied using time-ordered folds. Converging curves at a modest score indicate a bias / irreducible-noise ceiling; a persistent gap indicates variance that more data could reduce.

In [ ]:
sizes, tr_sc, va_sc = learning_curve(clone(MODELS[best_ml_id]), DATA[META[best_ml_id]['features']]['train'], y_train,
                                     train_sizes=np.linspace(0.1, 1.0, 6), cv=TSCV, scoring='roc_auc_ovr', n_jobs=-1)
plt.figure(figsize=(7, 4.5))
for sc, lab in [(tr_sc, 'training'), (va_sc, 'time-series CV')]:
    plt.plot(sizes, sc.mean(1), 'o-', label=lab)
    plt.fill_between(sizes, sc.mean(1) - sc.std(1), sc.mean(1) + sc.std(1), alpha=0.2)
plt.xlabel('Training examples'); plt.ylabel('Macro ROC-AUC'); plt.title(f'Learning curve: {best_ml_id}'); plt.legend()
plt.tight_layout(); plt.savefig(f'figures/learning_curve_{best_ml_id}.png', dpi=150); plt.show()

## 11. Error analysis, UCL subset and Manchester City case study
**11.1 Where does money stop working?** Matches are grouped by the size of the squad-value gap. If money "decides" games, accuracy should rise steeply with the gap, and upsets (the cheaper squad winning) should concentrate where the gap is small.

In [ ]:
eid = best_ml_id
err = test_df[['date', 'competition_id', 'is_ucl', 'value_diff', 'elo_diff', 'home_club_id', 'away_club_id', TARGET]].copy()
err['pred'] = P_TEST[eid].argmax(1)
err['correct'] = (err.pred == err[TARGET]).astype(int)
err['gap_bin'] = pd.qcut(err.value_diff.abs(), 5, labels=['Q1 (small gap)', 'Q2', 'Q3', 'Q4', 'Q5 (large gap)'])
rich_home = err.value_diff >= 0
err['upset'] = ((rich_home & (err[TARGET] == 0)) | (~rich_home & (err[TARGET] == 2))).astype(int)
gap_tbl = err.groupby('gap_bin').agg(matches=('correct', 'size'), accuracy=('correct', 'mean'),
                                     draw_rate=(TARGET, lambda s: (s == 1).mean()), upset_rate=('upset', 'mean')).round(3)
gap_tbl.to_csv('results/accuracy_by_value_gap.csv')
gap_tbl.plot(y=['accuracy', 'draw_rate', 'upset_rate'], marker='o', figsize=(8, 4), rot=0)
plt.title('Test accuracy, draw rate and upset rate by squad-value gap'); plt.ylabel('rate')
plt.tight_layout(); plt.savefig('figures/error_by_value_gap.png', dpi=150); plt.show()
gap_tbl

**11.2 Which features does the model rely on?** Permutation importance (drop in validation log-likelihood when a feature is shuffled).

In [ ]:
VI = np.random.RandomState(SEED).choice(len(y_val), size=min(10_000, len(y_val)), replace=False)
pi = permutation_importance(MODELS[best_ml_id], DATA[META[best_ml_id]['features']]['val'][VI], y_val[VI],
                            scoring='neg_log_loss', n_repeats=5, random_state=SEED, n_jobs=-1)
imp = pd.Series(pi.importances_mean, index=DATA[META[best_ml_id]['features']]['columns']).sort_values()
colors = ['#DD8452' if c in MONEY else '#4C72B0' for c in imp.index]
imp.plot(kind='barh', figsize=(7, 8), color=colors)
plt.title(f'Permutation importance ({best_ml_id}); orange = money features')
plt.tight_layout(); plt.savefig('figures/permutation_importance.png', dpi=150); plt.show()

**11.3 UCL subset.** Are Champions League matches more or less predictable than league matches? Elite clubs are closer in value, and two-legged knockout ties change incentives (a team protecting a lead may accept a draw).

In [ ]:
rows = []
for label, mask in [('All test matches', np.ones(len(err), bool)), ('Domestic leagues', test_df.is_european.values == 0),
                    ('UEFA Champions League', err.is_ucl.values == 1)]:
    if mask.sum() > 30 and len(np.unique(y_test[mask])) == 3:
        rows.append({'subset': label, 'matches': int(mask.sum()), **metrics_mc(y_test[mask], P_TEST[eid][mask])})
ucl_tbl = pd.DataFrame(rows).round(3)
ucl_tbl.to_csv('results/ucl_subset_results.csv', index=False)
ucl_tbl

**11.4 Manchester City case study.** Spending, squad value and win rate per season, alongside the model's test-period predictions for City's matches.

In [ ]:
city_ids = clubs.loc[clubs['name'].str.contains('Manchester City', case=False, na=False), 'club_id'].tolist() or [281]
city = long[long.club_id.isin(city_ids)].copy()
city['win'] = (city.points == 3).astype(int)
season_tbl = city.groupby('season').agg(matches=('win', 'size'), win_rate=('win', 'mean'),
                                        squad_value_m=('sq_value5', lambda s: s.mean() / 1e6),
                                        net_spend_1y_m=('net_spend_1y', lambda s: s.mean() / 1e6)).round(2)
fig, ax1 = plt.subplots(figsize=(10, 4.5))
ax1.bar(season_tbl.index.astype(str), season_tbl.squad_value_m, color='#6CABDD', label='Mean matchday squad value (€m)')
ax1.set_ylabel('€m'); ax1.tick_params(axis='x', rotation=45)
ax2 = ax1.twinx(); ax2.plot(season_tbl.index.astype(str), season_tbl.win_rate, 'o-', color='#1C2C5B', label='Win rate')
ax2.set_ylabel('Win rate'); ax2.set_ylim(0, 1)
fig.legend(loc='upper left', bbox_to_anchor=(0.08, 0.92)); plt.title('Manchester City: squad value vs win rate by season')
plt.tight_layout(); plt.savefig('figures/man_city_case_study.png', dpi=150); plt.show()

cm = err[err.home_club_id.isin(city_ids) | err.away_club_id.isin(city_ids)]
if len(cm):
    print(f"City test-period matches: {len(cm)} | model accuracy {cm.correct.mean():.3f} vs overall {err.correct.mean():.3f}")
season_tbl

**Interpretation (write after running).** Discuss: whether accuracy rises with the value gap and where upsets cluster; whether money features rank high in permutation importance or are overshadowed by Elo (which itself partly reflects past spending); whether UCL matches are harder to predict than league matches; and whether City's win rate tracks its squad value, keeping in mind that one club is an illustration, not proof.

## 12. Saving models and artefacts
The best traditional model is saved with `joblib` together with its scaler, feature order and competition index; the best deep model in `.keras` and `.h5`. Models and TensorBoard logs are zipped for the Google Drive folder.

In [ ]:
f = META[best_ml_id]['features']
joblib.dump({'model': MODELS[best_ml_id], 'scaler': DATA[f]['scaler'], 'scaled_cols': DATA[f]['scaled_cols'],
             'features': DATA[f]['columns'], 'classes': CLASS_NAMES, 'exp_id': best_ml_id},
            'models/best_traditional_model.joblib')
best_dl = MODELS[best_dl_id]
best_dl.save('models/best_dl_model.keras')
try:
    best_dl.save('models/best_dl_model.h5')
except Exception as e:
    print('H5 export skipped:', e)
f = META[best_dl_id]['features']
joblib.dump({'scaler': DATA[f]['scaler'], 'scaled_cols': DATA[f]['scaled_cols'], 'features': DATA[f]['columns'],
             'competition_index': COMP_INDEX, 'classes': CLASS_NAMES, 'exp_id': best_dl_id},
            'models/best_dl_preprocessing.joblib')
shutil.make_archive('models_bundle', 'zip', 'models')
shutil.make_archive('tensorboard_logs', 'zip', 'logs')
print(os.listdir('models'))

**Optional (Colab):** copy the raw data zip, models and logs to your Google Drive folder.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# DEST = '/content/drive/MyDrive/ML_Summative_Football'
# os.makedirs(DEST, exist_ok=True)
# for p in ['data/transfermarkt_raw.zip', 'models_bundle.zip', 'tensorboard_logs.zip']:
#     shutil.copy(p, DEST)

## 13. Conclusions
*(Complete after running, in your own words.)* Answer the research question directly: how much predictive power does money have on its own (ML-1), and how much does it add beyond Elo and form (ML-2 vs ML-3)? Which paradigm won on the test period, and was deep learning's extra complexity justified on this tabular data? Summarise the draw problem, where upsets occur, how UCL matches differ, and what the Manchester City case suggests. Limitations: Transfermarkt values are crowd-estimated, fees are missing for some transfers, wages (often the strongest financial predictor in the literature) are unavailable, and Elo is only loosely comparable across leagues. Future work: adding wage data, comparing against bookmaker odds, and ordinal or Poisson goal models.